Notebook to add extra columns to the summary csvs. 

Will add the simulation parameters as columns to the summary csvs.

In [1]:
import polars as pl
import os
from pathlib import Path
from sim_lib.sim_results import SimResults
import pandas as pd
from tqdm import tqdm
import shutil

In [2]:
DIR = Path("data/july_2025_combined")
NEW_DIR = Path("data/july_2025_combined_extra_cols")

NEW_DIR.mkdir(exist_ok=True)

In [3]:
combs = os.listdir(DIR)

In [6]:
def add_cols_to_comb(comb_dir: Path):
    sim = SimResults.load(comb_dir, False)

    data_params = sim.sim_params.data_params.__dict__
    model_params = sim.sim_params.model_params.__dict__

    params = {**data_params, **model_params}

    new_dir = NEW_DIR / comb_dir.name
    new_dir.mkdir(exist_ok=True)

    for param, summary in sim.summary.items():
        params_df = pd.DataFrame([params] * len(summary))
        new_summary = pd.concat([summary, params_df], axis=1)
        del new_summary["Unnamed: 0"]
        new_summary.to_csv(new_dir / f"{param}.csv")

In [7]:
for comb in tqdm(combs):
    add_cols_to_comb(DIR / comb)

100%|██████████| 3024/3024 [11:46<00:00,  4.28it/s]


In [8]:
full = []

new_combs = os.listdir(NEW_DIR)
for p in tqdm(new_combs):
    x = pl.read_csv(NEW_DIR / p / "beta0.csv")
    if len(x) > 50:
        full.append(p)
        new = Path("outputs/summary_with_extra_cols_full_oct_2_2025") / p
        # new.mkdir(exist_ok=True)
        shutil.copytree(NEW_DIR / p, new)

100%|██████████| 3024/3024 [02:09<00:00, 23.28it/s]


In [ ]:
full[0]